# 10 · 01 — Build: PM100 jobs ⋈ M100 ExaData telemetry

Session 08 asked whether *power* degrades before a job fails, and answered no. It could
not ask the question the thesis actually poses, because PM100 carries none of the eight
metrics the spec names (`docs/references/TCC_Dados_ML.pdf`, slide 8).

This session attaches the missing channels. PM100 is the exclusively-allocated subset of
M100, so its jobs own their sensors; M100 ExaData publishes the raw per-node streams for
the same cluster and the same months. `09_screening` verified that the two share a node
namespace and a clock, and measured the coverage: **median 0.909 for IPMI, 0.79 for
ganglia**.

Six months of telemetry (2020-05 .. 2020-10), the whole span PM100 covers.

**This notebook claims no pre-registration.** It builds the cohort and the series store
and asserts geometry. It computes nothing conditioned on the label — the first
label-aware number in this session appears in `02_degradation`, after the hypotheses are
written down.

In [1]:
import gc, json, os, sys, time
import numpy as np, pandas as pd

sys.path.insert(0, os.path.abspath("."))
import exalib as X

T_START, T_END = "2020-05-06", "2020-10-12 23:00"   # PM100's span, inside the telemetry
MIN_RUNTIME_S  = 1200        # one 20-minute window
os.makedirs("data", exist_ok=True)
print("months extracted:", X.months())
print("channels:", len(X.CHANNELS))

months extracted: ['20-05', '20-06', '20-07', '20-08', '20-09', '20-10']
channels: 19


## Step 1 — the cohort

PM100 jobs that both start and end inside the telemetry window, ran at least 20 minutes,
and are not `CANCELLED` (a user action, not an outcome). **No restriction on node count**
— every PM100 job is exclusive, so a 16-node job is as attributable as a 1-node job; the
telemetry is averaged over the nodes the job held.

In [2]:
J = X.load_jobs(T_START, T_END, MIN_RUNTIME_S)
print(f"cohort: {len(J):,} jobs, {J.user_id.nunique()} users")
print(J.job_state.value_counts().to_string())
print("\nnodes per job:", J.num_nodes_alloc.describe()[["mean", "50%", "max"]].to_dict())
print("wall h:", (J.wall / 3600).describe()[["mean", "50%", "max"]].round(2).to_dict())

cohort: 69,096 jobs, 392 users
job_state
COMPLETED        52967
TIMEOUT           7905
FAILED            7663
OUT_OF_MEMORY      418
NODE_FAIL          143

nodes per job: {'mean': 3.7605505383813824, '50%': 1.0, 'max': 256.0}
wall h: {'mean': 5.23, '50%': 1.41, 'max': 34.81}


## Step 2 — the series store

One row per `GRID` = 60 s per job per channel, averaged over the job's nodes. 60 s is
below every window we use (20 min = 20 rows) and above ganglia's slowest useful period,
so it costs no resolution the source has.

In [3]:
t0 = time.time()
flat, offs, cover = {}, {}, {}
for name in X.CHANNELS:
    tic = time.time()
    flat[name], offs[name] = X.build_series(name, J)
    gc.collect()
    cover[name] = float(np.isfinite(flat[name]).mean())
    print(f"  {name:12s} {len(flat[name]):>10,} rows  finite {cover[name]:6.1%}"
          f"  {time.time()-tic:6.1f}s", flush=True)
print(f"total {time.time()-t0:.0f}s")

  gpu0_temp    21,629,999 rows  finite  96.6%    73.4s


  gpu1_temp    21,629,999 rows  finite  96.6%    70.9s


  gpu3_temp    21,629,999 rows  finite  96.6%    62.1s


  gpu4_temp    21,629,999 rows  finite  96.6%    56.4s


  gpu0_mtemp   21,629,999 rows  finite  96.6%    57.3s


  gpu4_mtemp   21,629,999 rows  finite  96.6%    57.7s


... (10 outputs left out)

  proc_run     21,629,999 rows  finite  26.4%     8.6s


  cpu_wio      21,629,999 rows  finite  27.8%    10.3s


  load_one     21,629,999 rows  finite  30.0%    11.7s


total 771s


## Step 3 — derived channels

* `mem_active` = `mem_total − mem_free − mem_cached − mem_buffers`, in KB — this is the
  spec's `node_memory_Active_bytes`, the metric it names for memory pressure before an
  `OUT_OF_MEMORY` kill.
* `gpu_temp_max` = the hottest of the four GPU core sensors — overheating is a property
  of the worst device, not the average one.
* `gpu_pow` = the mean of the two `gv100card` sensors we hold.

In [4]:
def same_grid(*names):
    a = offs[names[0]]
    for n in names[1:]:
        assert np.array_equal(offs[n], a), n
    return a

g = same_grid(*X.CHANNELS)
offs["mem_active"] = g
flat["mem_active"] = (flat["mem_total"] - flat["mem_free"]
                      - flat["mem_cached"] - flat["mem_buf"]).astype(np.float32)
offs["gpu_temp_max"] = g
flat["gpu_temp_max"] = np.nanmax(np.vstack([flat["gpu0_temp"], flat["gpu1_temp"],
                                            flat["gpu3_temp"], flat["gpu4_temp"]]),
                                 axis=0).astype(np.float32)
offs["gpu_pow"] = g
flat["gpu_pow"] = np.nanmean(np.vstack([flat["gpu0_pow"], flat["gpu3_pow"]]),
                             axis=0).astype(np.float32)
for n in X.DERIVED:
    cover[n] = float(np.isfinite(flat[n]).mean())
    v = flat[n][np.isfinite(flat[n])]
    print(f"  {n:14s} finite {cover[n]:6.1%}  median {np.median(v):12.1f}")

C:\Users\fcall\AppData\Local\Temp\ipykernel_29648\2730522925.py:12: RuntimeWarning: All-NaN slice encountered
  flat["gpu_temp_max"] = np.nanmax(np.vstack([flat["gpu0_temp"], flat["gpu1_temp"],


C:\Users\fcall\AppData\Local\Temp\ipykernel_29648\2730522925.py:16: RuntimeWarning: Mean of empty slice
  flat["gpu_pow"] = np.nanmean(np.vstack([flat["gpu0_pow"], flat["gpu3_pow"]]),


  mem_active     finite  31.4%  median   28917522.0
  gpu_temp_max   finite  96.6%  median         47.3


  gpu_pow        finite  96.6%  median          0.0


## Step 4 — invariants

I1  every channel has the same per-job grid (so a window index means the same thing).
I2  a job's grid length matches its run time.
I3  `mem_active` is non-negative and below `mem_total`.
I4  temperatures are physical (0–120 °C) and `gpu_temp_max` dominates each sensor.
I5  the store round-trips.

In [5]:
K = np.diff(g)
assert (K >= MIN_RUNTIME_S // X.GRID).all(), "I2 short series"
assert np.allclose(K, J.wall.values // X.GRID, atol=1), "I2 length vs wall clock"
d = J.drift_s
print(f"I2 |wall - run_time|: median {d.median():.0f}s, p95 {d.quantile(.95):.0f}s, "
      f"max {d.max():.0f}s, >300s for {(d > 300).mean():.2%} of jobs")
ma = flat["mem_active"][np.isfinite(flat["mem_active"])]
mt = flat["mem_total"][np.isfinite(flat["mem_active"])]
print(f"I3 mem_active < 0: {(ma < 0).mean():.4%}   > mem_total: {(ma > mt).mean():.4%}")
gt = flat["gpu_temp_max"][np.isfinite(flat["gpu_temp_max"])]
print(f"I4 gpu_temp_max range {gt.min():.0f}..{gt.max():.0f} °C")
assert 0 <= gt.min() and gt.max() <= 120, "I4 non-physical temperature"
for s in ["gpu0_temp", "gpu1_temp", "gpu3_temp", "gpu4_temp"]:
    m = np.isfinite(flat[s]) & np.isfinite(flat["gpu_temp_max"])
    assert (flat["gpu_temp_max"][m] >= flat[s][m] - 1e-3).all(), f"I4 {s}"
print("I1/I2/I4 ok")

I2 |wall - run_time|: median 0s, p95 0s, max 0s, >300s for 0.00% of jobs
I3 mem_active < 0: 0.0000%   > mem_total: 0.0000%
I4 gpu_temp_max range 28..83 °C


I1/I2/I4 ok


In [6]:
np.savez_compressed("data/series.npz",
                    **{f"v__{k}": v for k, v in flat.items()},
                    **{f"o__{k}": v for k, v in offs.items()})
J.drop(columns=["nodes"]).to_parquet("data/jobs.parquet", index=False)
J[["job_id", "nodes"]].to_parquet("data/job_nodes.parquet", index=False)
z = np.load("data/series.npz")
assert np.array_equal(np.nan_to_num(z["v__gpu_temp_max"]),
                      np.nan_to_num(flat["gpu_temp_max"])), "I5 round-trip"
print("I5 ok —", os.path.getsize("data/series.npz") / 1e6, "MB")

I5 ok — 250.633023 MB


In [7]:
summary = {"t_start": T_START, "t_end": T_END, "min_runtime_s": MIN_RUNTIME_S,
           "grid_s": X.GRID, "n_jobs": int(len(J)), "n_users": int(J.user_id.nunique()),
           "states": J.job_state.value_counts().to_dict(),
           "coverage": {k: round(v, 4) for k, v in cover.items()}}
json.dump(summary, open("data/build_summary.json", "w"), indent=1)
print(json.dumps(summary["states"], indent=0))

{
"COMPLETED": 52967,
"TIMEOUT": 7905,
"FAILED": 7663,
"OUT_OF_MEMORY": 418,
"NODE_FAIL": 143
}
